# Bank of Canada Speech Hawkishness — Verification Notebook

Run this after completing the tournament and scoring pipeline.

**Pipeline:**
1. `scrape_speeches.py` → `data/speeches_raw.json`
2. `build_tournament.py` → `data/tournament_results.json`
3. `score_speeches.py` → `data/speeches_scored.csv`


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

df = pd.read_csv("data/speeches_scored.csv", parse_dates=["date"])
print(f"Loaded {len(df)} speeches")
print(f"Date range: {df.date.min().date()} to {df.date.max().date()}")
print(f"Speakers: {df.speaker.nunique()}")
df.head()

## 1. Tournament Convergence Check

In [ ]:
# All sigma should be < 2.0 (tournament convergence criterion)
unconverged = df[df.trueskill_sigma >= 2.0]
print(f"Unconverged speeches (sigma >= 2.0): {len(unconverged)}")
if len(unconverged) > 0:
    print("WARNING: Some speeches did not converge!")
    print(unconverged[["date", "speaker", "title", "trueskill_sigma"]].to_string())
else:
    print("All speeches converged. ✓")

print(f"
Comparisons per speech — min: {df.comparisons.min()}, mean: {df.comparisons.mean():.1f}, max: {df.comparisons.max()}")

## 2. Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["raw_score"], bins=30, color="steelblue", edgecolor="white", linewidth=0.5)
axes[0].set_title("Raw Score Distribution")
axes[0].set_xlabel("Raw Score (0–100)")
axes[0].set_ylabel("Count")
axes[0].axvline(50, color="red", linestyle="--", alpha=0.5, label="50 (neutral)")
axes[0].legend()

axes[1].hist(df["era_adjusted_score"], bins=30, color="darkorange", edgecolor="white", linewidth=0.5)
axes[1].set_title("Era-Adjusted Score Distribution")
axes[1].set_xlabel("Era-Adjusted Score (50 = neutral)")
axes[1].set_ylabel("Count")
axes[1].axvline(50, color="red", linestyle="--", alpha=0.5, label="50 (neutral)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Hawkishness Over Time

In [ ]:
# Rolling 12-month average of era-adjusted scores
df_sorted = df.sort_values("date").copy()
df_sorted = df_sorted.set_index("date")
rolling = df_sorted["era_adjusted_score"].rolling("365D").mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.scatter(df_sorted.index, df_sorted["era_adjusted_score"],
           alpha=0.3, s=15, color="steelblue", label="Individual speeches")
ax.plot(rolling.index, rolling.values, color="darkblue", linewidth=2, label="12-month rolling avg")
ax.axhline(50, color="red", linestyle="--", alpha=0.5, label="50 (neutral)")
ax.set_title("BoC Speech Hawkishness Over Time (Era-Adjusted)")
ax.set_xlabel("Date")
ax.set_ylabel("Era-Adjusted Score")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Speaker Rankings

In [ ]:
# Speaker averages (min 5 speeches)
speaker_stats = (
    df.groupby("speaker")["era_adjusted_score"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "avg_score", "std": "score_std", "count": "n_speeches"})
    .query("n_speeches >= 5")
    .sort_values("avg_score", ascending=False)
)

fig, ax = plt.subplots(figsize=(10, max(4, len(speaker_stats) * 0.4)))
colors = ["tomato" if v > 50 else "steelblue" for v in speaker_stats["avg_score"]]
ax.barh(speaker_stats.index, speaker_stats["avg_score"] - 50, color=colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Speaker Hawkishness (Era-Adjusted, min 5 speeches)")
ax.set_xlabel("Score relative to neutral (50)")
plt.tight_layout()
plt.show()

print(speaker_stats.to_string())

## 5. Most Hawkish and Dovish Speeches

In [ ]:
cols = ["date", "speaker", "title", "era_adjusted_score", "comparisons"]

print("TOP 10 MOST HAWKISH (era-adjusted):")
print(df.nlargest(10, "era_adjusted_score")[cols].to_string(index=False))

print()
print("TOP 10 MOST DOVISH (era-adjusted):")
print(df.nsmallest(10, "era_adjusted_score")[cols].to_string(index=False))